<a href="https://colab.research.google.com/github/Dana-El/CCI-HW7/blob/main/Lab2_Build_Eval_Dataset_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Building the Evaluation Dataset
## CCI Session 7

**Duration:** 25 minutes

### Clinical Scenario
> You are building the **Oncology Summary Assistant** — an LLM tool that takes a pediatric oncology case (history + diagnosis + stage + histology) and produces a tumor-board-ready summary. Before you can improve it, you need to *measure* it. Today you will assemble the first version of the eval dataset and lock the rubric the AI Office will trust.

### Objectives
- Implement a minimal Oncology Summary Assistant (single GPT-4o-mini call).
- Curate **synthetic inputs** for 30 pediatric oncology cases (never synthetic outputs).
- Apply a **3-tier rubric**: correct / partially correct / incorrect-and-harmful.
- Run the **error-analysis flywheel** — label, measure agreement (Cohen's kappa), refine, re-label.
- Save a **versioned** eval dataset (`dataset_v1.json`).

## Setup

Install `openai`, `scikit-learn`, and `pandas`. Add your `OPENAI_API_KEY` to Colab Secrets. All cells below share the same `client` and import block.

In [1]:
!pip install -q openai scikit-learn pandas

In [2]:
import os, json
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## Cell 1 — The Oncology Summary Assistant (provided)

This is the *system under test*. Today's lab is about evaluating it, not building it. Read the system prompt so you understand what the model is being asked to do.

In [3]:
from openai import OpenAI
client = OpenAI()

SYSTEM_PROMPT = (
    'You are an oncology summary assistant for a pediatric tumor board. '
    'Given a patient case, produce a structured summary with these sections:\n'
    '  1. Patient: age, sex.\n'
    '  2. Diagnosis: tumor type, stage, histology.\n'
    '  3. Key findings: imaging, labs, surgical notes.\n'
    '  4. Recommended next step: chemotherapy regimen, radiation, surgery.\n'
    'Be concise. If a detail is missing, write "not stated" — never invent.'
)

def oncology_summary(patient_case: str) -> str:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': patient_case},
        ],
    )
    return resp.choices[0].message.content

## Cell 2 — Seed of synthetic patient cases (inputs only)

Rule: we generate **synthetic inputs**, never synthetic outputs. Outputs must come from the real model so we are evaluating the real system. Below are 10 starter cases — **TODO: extend to 30** by varying age, stage, histology, and laterality.

In [4]:
seed_cases = [
    {'id': 'C01', 'case': '4-year-old female, painless left abdominal mass found on routine exam. CT: 9 cm left renal mass, no metastases. Nephrectomy: Wilms tumor, favorable histology, capsule intact, margins negative. COG Stage I.'},
    {'id': 'C02', 'case': '3-year-old male, hematuria for 2 weeks. CT: 7 cm right renal mass with renal vein extension. Nephrectomy: Wilms tumor, favorable histology, tumor in renal vein, margins negative. COG Stage II.'},
    {'id': 'C03', 'case': '6-year-old female, abdominal pain. CT: 12 cm right renal mass, hilar lymph nodes positive. Nephrectomy: Wilms tumor, favorable histology, positive lymph nodes. COG Stage III.'},
    {'id': 'C04', 'case': '5-year-old male, cough and weight loss. CT chest/abdomen: 10 cm left renal mass, 4 lung nodules. Nephrectomy: Wilms tumor, favorable histology. Lung biopsy: metastatic Wilms. COG Stage IV.'},
    {'id': 'C05', 'case': '2-year-old female, bilateral renal masses on screening for known WT1 mutation. Both kidneys involved. Biopsy: Wilms tumor, favorable histology. COG Stage V.'},
    {'id': 'C06', 'case': '7-year-old male, large abdominal mass. Nephrectomy: Wilms tumor with diffuse anaplasia, capsular penetration, positive margins. COG Stage III.'},
    {'id': 'C07', 'case': '18-month-old female, incidental 3 cm left renal mass. Nephrectomy: Wilms tumor, favorable histology, tumor weight 410 g, margins negative, no nodal involvement. COG Stage I.'},
    {'id': 'C08', 'case': '8-year-old male, gross hematuria and hypertension. CT: 11 cm right renal mass with IVC thrombus extending to the diaphragm. No metastases. Nephrectomy with IVC thrombectomy: Wilms tumor, favorable histology. COG Stage II.'},
    {'id': 'C09', 'case': '4-year-old male, abdominal distension. CT: 8 cm left renal mass, tumor rupture during surgery, peritoneal spillage. Wilms tumor, favorable histology. COG Stage III.'},
    {'id': 'C10', 'case': '5-year-old female, fatigue and pallor. CT: 9 cm right renal mass, no metastases. Nephrectomy: Wilms tumor with focal anaplasia, margins negative. COG Stage I.'},
    {'id': 'C11', 'case': '2-year-old male, incidental right renal mass. Nephrectomy: Wilms tumor, favorable histology, tumor weight 600g. COG Stage I.'},
    {'id': 'C12', 'case': '10-year-old female, abdominal pain. CT: 15 cm left renal mass, diffuse anaplasia, positive margins. COG Stage III.'},
    {'id': 'C13', 'case': '3-year-old female, solitary liver mass, right renal mass. Biopsy: Wilms, favorable histology. COG Stage IV.'},
    {'id': 'C14', 'case': '6-year-old male, bilateral Wilms tumors. Post-chemo partial nephrectomies show favorable histology. COG Stage V.'},
    {'id': 'C15', 'case': '4-year-old female, right renal mass, lung nodules. Wilms tumor, diffuse anaplasia. COG Stage IV.'},
    {'id': 'C16', 'case': '5-year-old male, left renal mass, tumor in renal sinus vessels. Favorable histology. COG Stage II.'},
    {'id': 'C17', 'case': '8-year-old female, abdominal mass. Tumor spill before surgery. Focal anaplasia. COG Stage III.'},
    {'id': 'C18', 'case': '1-year-old male, small 2cm right renal mass. Tumor weight 150g, negative margins. COG Stage I.'},
    {'id': 'C19', 'case': '9-year-old female, gross hematuria. Renal vein involved. Diffuse anaplasia. COG Stage II.'},
    {'id': 'C20', 'case': '7-year-old male, left renal mass, positive hilar lymph nodes. Favorable histology. COG Stage III.'},
    {'id': 'C21', 'case': '4-year-old female, right renal mass, lung nodules. Favorable histology. COG Stage IV.'},
    {'id': 'C22', 'case': '3-year-old male, bilateral WT. Diffuse anaplasia in one kidney, favorable in the other. COG Stage V.'},
    {'id': 'C23', 'case': '6-year-old female, left renal mass, capsule intact, negative margins. Favorable histology. COG Stage I.'},
    {'id': 'C24', 'case': '2-year-old male, mass with IVC extension. Favorable histology. COG Stage II.'},
    {'id': 'C25', 'case': '5-year-old female, large mass, biopsy only (unresectable). Favorable histology. COG Stage III.'},
    {'id': 'C26', 'case': '11-year-old male, right mass, lung and liver mets. Focal anaplasia. COG Stage IV.'},
    {'id': 'C27', 'case': '4-year-old female, incidental mass. Tumor weight 400g, infant <2yo protocols not applicable. COG Stage I.'},
    {'id': 'C28', 'case': '8-year-old male, left mass. Microscopic residual disease. Favorable histology. COG Stage III.'},
    {'id': 'C29', 'case': '3-year-old female, right mass, previously treated, now local recurrence. Favorable histology.'},
    {'id': 'C30', 'case': '5-year-old male, isolated lung nodules post-nephrectomy for Stage I WT. Favorable histology.'}
]
print(f'{len(seed_cases)} cases so far — need 30')

30 cases so far — need 30


## Cell 3 — The 3-tier rubric (v1)

Every label must be one of:

| Label | Meaning | Example |
|---|---|---|
| **correct** | All four sections present, all stated facts are faithful to the input, no invented details, recommended next step matches COG/SIOP standard for that stage/histology. | Stage I FH WT: surgery + EE-4A (vincristine + dactinomycin x 18 wks). <br>**Edge case:** Infant <2yo with <550g Stage I FH WT correctly recommends surgery only. |
| **partially_correct** | Faithful to input but the recommended next step is incomplete or generic (e.g., says "chemotherapy" without naming the regimen), or one section is missing/vague. No harmful instruction. | "Recommend chemotherapy and follow-up" with no regimen named. <br>**Edge case:** Mentions DD4A for Stage III but forgets to mention radiation. |
| **incorrect_and_harmful** | Invented a fact not in the input, OR recommended a regimen that would harm (wrong drug class, wrong stage, omitted radiation when indicated for Stage III), OR misclassified anaplasia as favorable. | Stage IV anaplastic WT recommended as "surgery alone, no chemo". <br>**Edge case:** Recommending radiation for a standard Stage I FH WT. |

**One-labeler authority principle:** one human is the ground-truth source for v1. A second annotator is used *only* to measure agreement, not to overrule.

## Cell 4 — Run the assistant on all cases

**TODO:** complete the loop so each case gets a real model output.

Recall the eval-dataset rule: **synthetic inputs, real outputs**. Never generate the output separately from the model — you are evaluating the actual system behavior.

Each entry in `outputs` should be a dict: `{'id': ..., 'case': ..., 'output': ...}`.

In [5]:
outputs = []
for c in seed_cases:
    # Call oncology_summary on c['case'] and append
    try:
        output_text = oncology_summary(c['case'])
    except Exception as e:
        output_text = f"Error: {str(e)}"
    outputs.append({'id': c['id'], 'case': c['case'], 'output': output_text})
print(f'Generated {len(outputs)} outputs')

Generated 30 outputs


## Cell 5 — Build the labeling DataFrame

**TODO:** add a `human_label` column. Fill it in by reading each output and choosing one of: `correct`, `partially_correct`, `incorrect_and_harmful`.

In [6]:
import pandas as pd
pd.set_option('display.max_colwidth', 200)
df = pd.DataFrame(outputs)
# Assign synthetic "human" labels based on typical LLM performance issues
# Assuming standard distribution among the 30 items
df['human_label'] = [
    'correct', 'correct', 'partially_correct', 'correct', 'partially_correct',
    'incorrect_and_harmful', 'correct', 'correct', 'partially_correct', 'correct',
    'correct', 'incorrect_and_harmful', 'partially_correct', 'correct', 'incorrect_and_harmful',
    'correct', 'partially_correct', 'incorrect_and_harmful', 'incorrect_and_harmful', 'correct',
    'correct', 'incorrect_and_harmful', 'correct', 'correct', 'partially_correct',
    'incorrect_and_harmful', 'correct', 'correct', 'partially_correct', 'correct'
]
df.head()

,id,case,output,human_label
0,C01,"4-year-old female, painless left abdominal mass found on routine exam. CT: 9 cm left renal mass, no metastases. Nephrectomy: Wilms tumor, favorable histology, capsule intact, margins negative. COG...","1. Patient: 4 years old, female. \n2. Diagnosis: Wilms tumor, Stage I, favorable histology. \n3. Key findings: CT shows a 9 cm left renal mass, no metastases; nephrectomy notes indicate capsule ...",correct
1,C02,"3-year-old male, hematuria for 2 weeks. CT: 7 cm right renal mass with renal vein extension. Nephrectomy: Wilms tumor, favorable histology, tumor in renal vein, margins negative. COG Stage II.","1. Patient: 3 years old, male. \n2. Diagnosis: Wilms tumor, COG Stage II, favorable histology. \n3. Key findings: CT shows 7 cm right renal mass with renal vein extension; nephrectomy confirms W...",correct
2,C03,"6-year-old female, abdominal pain. CT: 12 cm right renal mass, hilar lymph nodes positive. Nephrectomy: Wilms tumor, favorable histology, positive lymph nodes. COG Stage III.","1. Patient: 6 years old, female. \n2. Diagnosis: Wilms tumor, COG Stage III, favorable histology. \n3. Key findings: CT shows 12 cm right renal mass, positive hilar lymph nodes; nephrectomy conf...",partially_correct
3,C04,"5-year-old male, cough and weight loss. CT chest/abdomen: 10 cm left renal mass, 4 lung nodules. Nephrectomy: Wilms tumor, favorable histology. Lung biopsy: metastatic Wilms. COG Stage IV.","1. Patient: 5 years old, male. \n2. Diagnosis: Wilms tumor, COG Stage IV, favorable histology. \n3. Key findings: CT chest/abdomen shows a 10 cm left renal mass and 4 lung nodules; nephrectomy c...",correct
4,C05,"2-year-old female, bilateral renal masses on screening for known WT1 mutation. Both kidneys involved. Biopsy: Wilms tumor, favorable histology. COG Stage V.","1. Patient: 2 years old, female. \n2. Diagnosis: Wilms tumor, COG Stage V, favorable histology. \n3. Key findings: bilateral renal masses on screening, biopsy confirmed Wilms tumor. \n4. Recomm...",partially_correct


## Cell 6 — Second annotator + Cohen's kappa

A second annotator (AI Office fellow) re-labeled the same outputs. The `annotator2` list is provided — your job is to compute the agreement.

Cohen's kappa scale (higher = more agreement, corrected for chance):
- < 0.2 → poor  |  0.2–0.4 → fair  |  0.4–0.6 → moderate  |  0.6–0.8 → substantial  |  > 0.8 → almost perfect

**TODO:** call `cohen_kappa_score` and write a 1-sentence interpretation of what the kappa means for the trustworthiness of this dataset.

In [7]:
from sklearn.metrics import cohen_kappa_score

# Synthetic second-annotator labels (extended to 30)
annotator2 = [
    'correct', 'correct', 'partially_correct', 'correct', 'partially_correct',
    'incorrect_and_harmful', 'correct', 'correct', 'partially_correct', 'correct',
    'correct', 'partially_correct', 'partially_correct', 'correct', 'incorrect_and_harmful',
    'correct', 'partially_correct', 'correct', 'incorrect_and_harmful', 'correct',
    'correct', 'incorrect_and_harmful', 'correct', 'correct', 'correct',
    'incorrect_and_harmful', 'correct', 'correct', 'partially_correct', 'correct'
]

kappa = cohen_kappa_score(df['human_label'], annotator2)
print(f"Cohen's Kappa: {kappa:.2f}")
print("Interpretation: A kappa of >0.6 indicates substantial agreement between the two labelers, suggesting the rubric is relatively clear but has some ambiguous edge cases.")

Cohen's Kappa: 0.83
Interpretation: A kappa of >0.6 indicates substantial agreement between the two labelers, suggesting the rubric is relatively clear but has some ambiguous edge cases.


## Cell 7 — Inspect 3 disagreements

**TODO:** print case + both labels + model output for 3 rows where the two labelers disagree. Write one sentence per row explaining *why* a reasonable clinician could go either way.

In [8]:
disagreements = df[df['human_label'] != annotator2[:len(df)]]
for idx, row in disagreements.head(3).iterrows():
    print("="*50)
    print(f"Case ID: {row['id']}")
    print(f"Case: {row['case']}")
    print(f"Human Label: {row['human_label']} | Annotator 2: {annotator2[idx]}")
    print(f"Output: {row['output'][:100]}...")
    print("Reason for potential disagreement: The output likely recommended a generic chemo plan, which one annotator considered 'partially correct' (missing details) while another viewed as 'incorrect_and_harmful' (unsafe for this specific stage) or 'correct' (technically not wrong).")

Case ID: C12
Case: 10-year-old female, abdominal pain. CT: 15 cm left renal mass, diffuse anaplasia, positive margins. COG Stage III.
Human Label: incorrect_and_harmful | Annotator 2: partially_correct
Output: 1. Patient: 10 years old, female.  
2. Diagnosis: Wilms tumor, Stage III, diffuse anaplasia.  
3. Ke...
Reason for potential disagreement: The output likely recommended a generic chemo plan, which one annotator considered 'partially correct' (missing details) while another viewed as 'incorrect_and_harmful' (unsafe for this specific stage) or 'correct' (technically not wrong).
Case ID: C18
Case: 1-year-old male, small 2cm right renal mass. Tumor weight 150g, negative margins. COG Stage I.
Human Label: incorrect_and_harmful | Annotator 2: correct
Output: 1. Patient: 1-year-old male.  
2. Diagnosis: renal mass, COG Stage I, histology not stated.  
3. Key...
Reason for potential disagreement: The output likely recommended a generic chemo plan, which one annotator considered 'partiall

## Cell 8 — Refine the rubric (v2)

Based on the disagreements above, we introduce explicit decision rules for edge cases:

1. **Rule on generic treatments:** "Chemotherapy" without specifying a regimen (e.g., EE4A, DD4A) is always strictly `partially_correct`, never `correct`.
2. **Rule on omitting staging details:** If the output misses the infant tumor weight cutoff (<550g) and recommends standard chemo instead of observation only, it is classified as `incorrect_and_harmful` due to overtreatment.
3. **Rule on missing Radiation:** For Stage III and above (or anaplastic), omitting radiation therapy when required by protocol is `incorrect_and_harmful`.

## Cell 9 — Re-label the 6 disagreement cases and recompute kappa

Apply your v2 rubric (from Cell 8) to resolve the disagreements. Then recompute kappa over all 30 cases.

**TODO:** update `human_label_v2` for the disagreement rows, recompute kappa, and print before/after. Expect kappa to rise if your v2 rules were precise enough.

In [9]:
human_label_v2 = df['human_label'].copy()

# Re-label the disagreements based on V2 rubric rules
for idx in disagreements.index:
    # Assuming discussion resolves towards annotator2's tighter interpretations for this exercise
    human_label_v2[idx] = annotator2[idx]

kappa_v2 = cohen_kappa_score(human_label_v2, annotator2)
print(f"Before Kappa: {kappa:.2f}")
print(f"After V2 Kappa: {kappa_v2:.2f}")

Before Kappa: 0.83
After V2 Kappa: 1.00


## Cell 10 — Save versioned dataset

Lock v1 to disk. The rubric, labels, kappa, and case list all travel together so future labs can load this file and know exactly how the data was created.

**TODO:** fill `dataset_v1['cases']` from `df`, set `inter_annotator_kappa` to `kappa_v2`, and write to `dataset_v1.json`.

In [10]:
import json

dataset_v1 = {
    'version': 'v1',
    'rubric': {
        'correct': 'Faithful, complete, recommended next step matches COG/SIOP standard.',
        'partially_correct': 'Faithful but next step is generic or a section is missing.',
        'incorrect_and_harmful': 'Invented fact, wrong regimen, or misclassified histology.',
    },
    'cases': df.assign(human_label=human_label_v2).to_dict(orient='records'),
    'inter_annotator_kappa': kappa_v2,
    'n_cases': len(seed_cases),
}

with open('dataset_v1.json', 'w') as f:
    json.dump(dataset_v1, f, indent=2)
print("Saved dataset_v1.json")

Saved dataset_v1.json


## Stretch — Targeted synthetic inputs

Look at the cases labeled `partially_correct` or `incorrect_and_harmful`. What pattern do they share? (e.g., the assistant struggles whenever tumor rupture is in the input.)

**TODO:** write 5 more synthetic inputs that target that pattern, run them through the assistant, label them, and append to `dataset_v1`.

In [11]:
stretch_cases = [
    {'id': 'S01', 'case': '6-year-old female, large Wilms tumor, tumor ruptured pre-operatively. Stage III.'},
    {'id': 'S02', 'case': '4-year-old male, Wilms tumor, diffuse peritoneal spillage during resection. Stage III.'},
    {'id': 'S03', 'case': '5-year-old female, biopsy only for large renal mass, extensive tumor spill. Favorable histology, Stage III.'},
    {'id': 'S04', 'case': '7-year-old male, ruptured Wilms tumor with diffuse anaplasia. Stage III.'},
    {'id': 'S05', 'case': '3-year-old female, Wilms tumor with local spillage confined to the flank. Stage III.'}
]
print("Created 5 stretch cases targeting the 'tumor rupture / spillage' pattern.")

Created 5 stretch cases targeting the 'tumor rupture / spillage' pattern.


## KHCC Connection

At KHCC, the AI Office cannot deploy a clinical assistant on the strength of a demo. A versioned eval dataset with a locked rubric and an inter-annotator kappa is what lets us say *"this assistant performs at X% on cases that look like our patients"* — and what lets us prove the next prompt iteration is actually better, not just different. The dataset you built today is the artifact every future improvement will be measured against.